# Valley Fever | Checkpoint 2: Tier 1 → Tier 2, all three counties

**One Colab notebook + one Excel input.** This continues the team's original combined Tier 1 run and then executes the final Tier 2 validation from the ZIP supplied on November 3, 2026. **No AZMet/EPA/PRISM re-download is needed.**

**Input:** `VF_Checkpoint2_Input_Three_Counties.xlsx` (six sheets, including 396 `COUNTY_MONTH` and 357 `FORECAST_READY` rows).  
**Output:** `VF_Checkpoint2_Tier1_Tier2_Results.xlsx` + `VF_Checkpoint2_All_Results.zip` (Tier 1 CSVs, final Tier 2 CSVs/charts, notes).

**Run:** Google Colab → File → Upload notebook → Runtime → Run all → upload the Excel once when asked. Let the full notebook finish. The last cell downloads the combined ZIP.

### Presentation scope
- **Checkpoint 1 recap:** Four data sources, cleaning, county-month integration, seasonal baseline and preliminary Ridge demo.
- **Checkpoint 2:** Reproduce combined Tier 1 results, development-only nested selection, final NB2/XGBoost comparison, full-2024 vs April–December sensitivity, SHAP, lag/delay tests and limitations.
- **Not included:** Tier 3 spatial GNN/STGCN; no ZIP-level cases or trained spatial graph in the supplied package.

**Limits:** 2024 was inspected during prior experiments, so do not describe it as a never-before-seen confirmatory test across the entire project. January–March 2024 counts are unchanged despite the documented test-kit concern; 2025 is provisional. PRISM here uses county centroids, AZMet is a station proxy, PM10 is a dust proxy, and historical data release timing is not established.

In [ ]:
# STEP 1 — Check the libraries used in the original Tier 1 and Tier 2 notebooks.
import importlib.util, subprocess, sys
packages = {
    'numpy':'numpy','pandas':'pandas','openpyxl':'openpyxl',
    'sklearn':'scikit-learn','xgboost':'xgboost','statsmodels':'statsmodels',
    'shap':'shap','matplotlib':'matplotlib','xlsxwriter':'XlsxWriter'
}
missing=[pkg for module,pkg in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print('Installing missing packages:',missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
else:
    print('All required packages already installed.')

In [ ]:
# STEP 2 — Upload ONE input Excel workbook, not the full original ZIP.
from pathlib import Path
NAME = 'VF_Checkpoint2_Input_Three_Counties.xlsx'
if not Path(NAME).exists():
    try:
        from google.colab import files
        print('Upload:', NAME)
        uploaded = files.upload()
        if NAME not in uploaded:
            raise FileNotFoundError('Please upload '+NAME+' with its exact filename.')
    except ImportError:
        raise FileNotFoundError('Place '+NAME+' beside the notebook and run again.')
print('Excel input ready:',Path(NAME).resolve(),round(Path(NAME).stat().st_size/1024),'KB')

## Part A — Tier 1: reproduce the previous checkpoint's combined models

The next cells reuse the original combined Tier 1 modeling logic: **seasonal naive, Ridge, and nonlinear boosting**, with 2019–2023 walk-forward validation, 2024 retrospective test and 2025 provisional evaluation. This is the same workflow as `VF_ThreeCounty_Colab.ipynb` in your ZIP, now reading the single Excel input above.

The separate Maricopa-only Ridge demo had different features and results. Do **not** mix its 133.71 MAE with the three-county Tier 1 results below.

In [ ]:
# Cell 3 - load the two key Excel sheets and validate our merge.
import numpy as np
import pandas as pd
import calendar
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from IPython.display import display

month = pd.read_excel(NAME, sheet_name='COUNTY_MONTH', dtype={'County_FIPS': str, 'Month': str})
forecast = pd.read_excel(NAME, sheet_name='FORECAST_READY',
                         dtype={'County_FIPS': str, 'Feature_Month': str, 'Target_Month': str})
expected_counties = {'Maricopa', 'Pinal', 'Pima'}
assert len(month) == 396, f'Expected 396 county-month rows; got {len(month)}'
assert len(forecast) == 357, f'Expected 357 forecast rows; got {len(forecast)}'
assert set(month['County']) == set(forecast['County']) == expected_counties
assert not month.duplicated(['County_FIPS', 'Month']).any(), 'Repeated county-month keys'
assert not forecast.duplicated(['County_FIPS', 'Target_Month']).any(), 'Repeated forecast targets'

month['County_FIPS'] = month['County_FIPS'].str.zfill(5)
forecast['County_FIPS'] = forecast['County_FIPS'].str.zfill(5)
expected_fips = {'Maricopa':'04013','Pinal':'04021','Pima':'04019'}
for c, fips in expected_fips.items():
    rows = month.loc[month['County'] == c]
    assert len(rows) == 132 and set(rows['County_FIPS']) == {fips}
    assert len(forecast.loc[forecast['County'] == c]) == 119

# Confirm each target truly matches the event-date case count of the following month.
lookup = month.set_index(['County_FIPS', 'Month'])['VF_cases_event_date']
for r in forecast.itertuples(index=False):
    target = int(lookup.loc[(r.County_FIPS, r.Target_Month)])
    assert int(r.Target_Cases_Next_Month) == target, (r.County, r.Target_Month)
    assert (pd.Period(r.Feature_Month, freq='M') + 1) == pd.Period(r.Target_Month, freq='M')

# Do not mix a future year's event-date outcome into the features for any target.
assert not any(c.startswith('Target_Month_') for c in forecast.columns)
print('PASS: 396 county-month records, 357 forecasts, 3 counties, unique keys and correct targets.')
print('Case totals by county / year:')
month['Year'] = month['Month'].str[:4].astype(int)
display(month.loc[month['Year'].isin([2024, 2025])].pivot_table(index='County', columns='Year', values='VF_cases_event_date', aggfunc='sum'))
print('Forecast splits:')
display(forecast.groupby(['Split','County']).size().unstack(fill_value=0))
# Preserve the 55-column canonical Tier 2 input BEFORE adding Tier 1 modeling columns.
model_input = forecast.copy(deep=True)
assert model_input.shape == (357,55), model_input.shape


In [ ]:
# Cell 4 - choose consistent predictors and build simple, leakage-aware models.
FEATURE_GROUPS = [
    'PRISM_rain_total_in', 'PRISM_tmean_F', 'EPA_PM10_mean_ugm3',
    'AZMET_soil4_C', 'AZMET_soil20_C', 'AZMET_RH_pct',
    'AZMET_wind_mean_ms', 'AZMET_wind_u_ms', 'AZMET_wind_v_ms', 'AZMET_VPD_kPa'
]
LAGS = ['current','lag1','lag2','lag3']
weather_cols = [f'{name}_{lag}' for name in FEATURE_GROUPS for lag in LAGS]
case_cols = ['VF_cases_current','VF_cases_lag1','VF_cases_lag2','Seasonal_Baseline_Cases']
input_cols = case_cols + weather_cols + ['month_sin','month_cos', 'County_Maricopa','County_Pinal','County_Pima']
missing_cols = sorted(set(case_cols + weather_cols) - set(forecast.columns))
assert not missing_cols, 'Missing predictors: '+str(missing_cols)

forecast['target_year'] = forecast['Target_Month'].str[:4].astype(int)
forecast['target_month_num'] = forecast['Target_Month'].str[5:7].astype(int)
forecast['month_sin'] = np.sin(2*np.pi*forecast['target_month_num']/12)
forecast['month_cos'] = np.cos(2*np.pi*forecast['target_month_num']/12)
for c in expected_counties:
    forecast[f'County_{c}'] = (forecast['County'] == c).astype(int)
for col in case_cols + weather_cols:
    forecast[col] = pd.to_numeric(forecast[col], errors='coerce')
forecast['Target_Cases_Next_Month'] = pd.to_numeric(forecast['Target_Cases_Next_Month'],errors='raise')

print('Missing predictor cells:', int(forecast[input_cols].isna().sum().sum()))
# Median filling occurs INSIDE the training pipeline, never using future evaluation data.
MODELS = {
  'Seasonal naive': None,
  'Ridge': make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=30)),
  'Nonlinear boosting': make_pipeline(SimpleImputer(strategy='median'),
      HistGradientBoostingRegressor(max_iter=110, learning_rate=.06,
                                   max_leaf_nodes=10,min_samples_leaf=15,random_state=42))
}

def predict_models(train, test, experiment):
    assert set(train['Target_Month']).isdisjoint(set(test['Target_Month']))
    assert train['Target_Month'].max() < test['Target_Month'].min(), 'Time leakage in split'
    Xtr, Xte = train[input_cols].to_numpy(dtype=float), test[input_cols].to_numpy(dtype=float)
    ytr = train['Target_Cases_Next_Month'].to_numpy(dtype=float)
    outputs=[]
    for name, estimator in MODELS.items():
        if estimator is None:
            prediction = test['Seasonal_Baseline_Cases'].to_numpy(dtype=float)
        else:
            model = clone(estimator)
            model.fit(Xtr, ytr)
            prediction = model.predict(Xte)
        prediction = np.maximum(0,np.asarray(prediction,dtype=float))
        for r,pred in zip(test.itertuples(index=False),prediction):
            outputs.append({'Experiment':experiment, 'Model':name, 'County':r.County,
                            'Target_Month':r.Target_Month, 'Actual_Cases':int(r.Target_Cases_Next_Month),
                            'Predicted_Cases':float(pred), 'Abs_Error':abs(float(pred)-float(r.Target_Cases_Next_Month)),
                            '2024_Early_Concern':r.Early_2024_Surveillance_Flag})
    return pd.DataFrame(outputs)

def metrics(df, period):
    res=[]
    for (model,county), g in df.groupby(['Model','County'],sort=True):
        y=g.Actual_Cases.to_numpy(dtype=float)
        p=g.Predicted_Cases.to_numpy(dtype=float)
        res.append({'Period':period,'Model':model,'County':county,'Months':len(g),
                    'MAE':round(mean_absolute_error(y,p),2),
                    'RMSE':round(np.sqrt(mean_squared_error(y,p)),2),
                    'WAPE_pct':round(100*np.abs(y-p).sum()/max(y.sum(),1),2)})
    return pd.DataFrame(res).sort_values(['County','MAE'])

In [ ]:
# Cell 5 - expanding-window development validation, year by year (2019–2023).
# No 2024 or 2025 target rows are used to select or validate models here.
parts=[]
for year in range(2019,2024):
    train = forecast.loc[forecast['target_year'] < year].copy()
    test  = forecast.loc[forecast['target_year'] == year].copy()
    parts.append(predict_models(train, test, f'Walkforward_{year}'))
walkforward = pd.concat(parts,ignore_index=True)
print('Walk-forward monthly forecast rows:',len(walkforward))
walkforward_metrics = metrics(walkforward,'2019-2023 development walk-forward')
display(walkforward_metrics)

In [ ]:
# Cell 6 - 2024 held-out test. Fit through Dec 2023 only, not 2024.
train_2024 = forecast.loc[forecast['target_year'] <= 2023].copy()
test_2024 = forecast.loc[forecast['target_year'] == 2024].copy()
result_2024 = predict_models(train_2024,test_2024,'2024 held-out')
print('Full 2024, all 12 target months:')
full_2024_metrics = metrics(result_2024,'2024 Jan-Dec')
display(full_2024_metrics)

print('2024 sensitivity check - April through December only (same trained models):')
late_2024 = result_2024.loc[result_2024['Target_Month']>='2024-04'].copy()
late_2024_metrics = metrics(late_2024, '2024 Apr-Dec only')
display(late_2024_metrics)

# Importantly, Jan–Mar remain in the original data and full-year evaluation.
assert len(result_2024)==3*3*12
assert len(late_2024)==3*3*9

In [ ]:
# Cell 7 - 2025 provisional evaluation is reported separately.
# Refit using observed target months through 2024; 2024 lab-count concerns remain a limitation.
train_2025 = forecast.loc[forecast['target_year'] <= 2024].copy()
test_2025 = forecast.loc[forecast['target_year'] == 2025].copy()
result_2025 = predict_models(train_2025,test_2025,'2025 provisional')
provisional_metrics = metrics(result_2025,'2025 provisional Jan-Dec')
display(provisional_metrics)

all_metrics = pd.concat([walkforward_metrics,full_2024_metrics,late_2024_metrics,provisional_metrics],ignore_index=True)
all_predictions = pd.concat([walkforward,result_2024,result_2025],ignore_index=True)
print('Retrospective evaluation complete. Compare per-county MAE/WAPE, not just pooled case counts.')

In [ ]:
# STEP 3 — Save Tier 1 results and hand off the exact same forecast-ready rows to Tier 2.
from pathlib import Path
import zipfile
TIER1_DIR=Path('VF_Checkpoint2_Tier1_CSVs')
TIER1_DIR.mkdir(exist_ok=True)
tier1_files={
    'Tier1_Model_Metrics.csv':all_metrics,
    'Tier1_Development_Walkforward.csv':walkforward,
    'Tier1_2024_Test.csv':result_2024,
    'Tier1_2024_AprDec_Sensitivity.csv':late_2024,
    'Tier1_2025_Provisional.csv':result_2025,
}
for filename,table in tier1_files.items():
    table.to_csv(TIER1_DIR/filename,index=False)
MODEL_CSV=Path('VF_ThreeCounty_Stage2_Model_Input.csv')
model_input.to_csv(MODEL_CSV,index=False)
assert model_input.shape==(357,55)
assert set(model_input.County)=={'Maricopa','Pinal','Pima'}
print('Tier 1 complete. Tier 2 input built from the SAME Excel:',MODEL_CSV)
print('2024 Tier 1 results:')
print(all_metrics[all_metrics.Period=='2024 Jan-Dec'].to_string(index=False))

## Part B — Tier 2: final frozen model selection and evaluation

The next two code cells are carried over from `FINAL_TIER2/VF_Tier2_Final.ipynb` in your ZIP. They fit county-only and pooled models, use **2020–2023 nested chronological validation**, choose the final models from development data, then evaluate 2024 and provisional 2025. They also produce SHAP, environmental lag tests, simulated reporting delays and sensitivity results.

**Selected models in the saved run:** Maricopa NB2, Pinal pooled XGB_medium, Pima pooled XGB_regularized. The archived 2026-10-15 results selected NB2, XGB_medium, XGB_regularized. **The original ZIP did not pin software versions.** A new run can select a different setting and produce noticeably different MAEs if the library environment changes. Compare results within one run; do not choose a model using 2024 outcomes.

In [ ]:
import os, warnings, math, json, time
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial

SEED = 42
INPUT_FILE='VF_ThreeCounty_Stage2_Model_Input.csv'
OUTPUT_DIR='VF_Tier2_Final_Outputs'
COUNTIES=['Maricopa','Pinal','Pima']
# Low-complexity pre-declared XGBoost grid. No 2024-based tuning.
XGB_GRID={
 'XGB_small': dict(n_estimators=130,max_depth=2,learning_rate=0.05,min_child_weight=5,subsample=0.85,colsample_bytree=0.85,reg_lambda=12),
 'XGB_medium': dict(n_estimators=190,max_depth=3,learning_rate=0.04,min_child_weight=5,subsample=0.80,colsample_bytree=0.80,reg_lambda=15),
 'XGB_regularized': dict(n_estimators=100,max_depth=2,learning_rate=0.07,min_child_weight=10,subsample=0.90,colsample_bytree=0.90,reg_lambda=25),
}
CANDIDATES=['Seasonal','Poisson','NB2']+list(XGB_GRID)
CORE_ENV=['PRISM_rain_total_in','EPA_PM10_mean_ugm3','AZMET_soil20_C','AZMET_RH_pct','AZMET_wind_mean_ms','AZMET_VPD_kPa']
FEATURE_BASE=['case_log','seasonal_log','month_sin','month_cos','rain_log','pm10','soil20','rh','wind','vpd']
ISSUES=[]

def read_input(path=INPUT_FILE):
    d=pd.read_csv(path,dtype={'County_FIPS':str}).copy()
    req=['County','Feature_Month','Target_Month','Target_Cases_Next_Month','Seasonal_Baseline_Cases',
         'VF_cases_current','VF_cases_lag1','VF_cases_lag2','Source_geography_note']
    for suffix in ['current','lag1','lag2']:
        req += [p+'_'+suffix for p in CORE_ENV]
    missing=[c for c in req if c not in d.columns]
    if missing: raise ValueError('Missing source columns: '+str(missing))
    d['Feature_dt']=pd.to_datetime(d.Feature_Month,format='%Y-%m')
    d['Target_dt']=pd.to_datetime(d.Target_Month,format='%Y-%m')
    d['Year']=d.Target_dt.dt.year.astype(int)
    d['Month']=d.Target_dt.dt.month.astype(int)
    if not ((d.Feature_dt+pd.offsets.MonthBegin(1))==d.Target_dt).all(): raise ValueError('Forecast dates are not strictly next-month.')
    if d.duplicated(['County','Target_Month']).any():raise ValueError('Duplicate county-month keys.')
    if sorted(d.County.unique().tolist())!=sorted(COUNTIES): raise ValueError('Unexpected county list')
    if d.groupby('County').size().to_dict()!={k:119 for k in COUNTIES}: raise ValueError('Expected 119 months/county.')
    if not (d.Target_Cases_Next_Month>=0).all(): raise ValueError('Negative case totals.')
    if (d.Year<2016).any() or (d.Year>2025).any():raise ValueError('Unexpected year')
    needed=[c for c in req if c in ['Target_Cases_Next_Month','Seasonal_Baseline_Cases','VF_cases_current','VF_cases_lag1','VF_cases_lag2'] or any(c.startswith(base+'_') for base in CORE_ENV)]
    for col in needed:
        d[col]=pd.to_numeric(d[col],errors='raise')
        if not np.isfinite(d[col]).all():raise ValueError('Missing/inf in model input: '+col)
    d=d.sort_values(['Year','Month','County']).reset_index(drop=True)
    return d

def features(d,case_delay=0, weather_delay=0, lag_mode='single', pooled=False):
    if case_delay not in (0,1,2):raise ValueError('case delay must be 0,1,2')
    if weather_delay not in (0,1,2):raise ValueError('weather delay must be 0,1,2')
    idx='current' if case_delay==0 else 'lag'+str(case_delay)
    widx='current' if weather_delay==0 else 'lag'+str(weather_delay)
    df=pd.DataFrame(index=d.index)
    df['case_log']=np.log1p(d['VF_cases_'+idx].astype(float).clip(lower=0))
    df['seasonal_log']=np.log1p(d.Seasonal_Baseline_Cases.astype(float).clip(lower=0))
    df['month_sin']=np.sin(2*np.pi*d.Month/12)
    df['month_cos']=np.cos(2*np.pi*d.Month/12)
    def add_env(sfx, post=''):
        df['rain_log'+post]=np.log1p(d['PRISM_rain_total_in_'+sfx].astype(float).clip(lower=0))
        df['pm10'+post]=d['EPA_PM10_mean_ugm3_'+sfx].astype(float)
        df['soil20'+post]=d['AZMET_soil20_C_'+sfx].astype(float)
        df['rh'+post]=d['AZMET_RH_pct_'+sfx].astype(float)
        df['wind'+post]=d['AZMET_wind_mean_ms_'+sfx].astype(float)
        df['vpd'+post]=d['AZMET_VPD_kPa_'+sfx].astype(float)
    add_env(widx)
    if lag_mode=='plus_lag1':
        later='lag1' if weather_delay==0 else 'lag2'
        add_env(later,'_extra')
    elif lag_mode!='single':raise ValueError(lag_mode)
    if pooled:
        df['is_maricopa']=(d.County=='Maricopa').astype(float)
        df['is_pinal']=(d.County=='Pinal').astype(float)
    if df.isna().any().any() or not np.isfinite(df.to_numpy()).all():raise ValueError('NA/inf feature in scenario')
    return df

def _design(t,v):
    X=features(t);Y=features(v)
    sc=StandardScaler()
    x=sc.fit_transform(X)
    z=sc.transform(Y)
    return sm.add_constant(x,has_constant='add'),sm.add_constant(z,has_constant='add')

def _safe_count(p, label):
    p=np.asarray(p,dtype=float)
    if not np.isfinite(p).all():raise ValueError('Nonfinite prediction for '+label)
    return np.maximum(p,0.)

def fit_predict(train,valid,model,case_delay=0,weather_delay=0,lag_mode='single',pooled=False):
    if model=='Seasonal':
        return _safe_count(valid.Seasonal_Baseline_Cases.to_numpy(),model),{'alpha':np.nan,'dispersion_method':'NA'}
    xtrain=features(train,case_delay,weather_delay,lag_mode,pooled)
    xvalid=features(valid,case_delay,weather_delay,lag_mode,pooled)
    y=train.Target_Cases_Next_Month.to_numpy(dtype=float)
    if model.startswith('XGB'):
        params=XGB_GRID[model]
        reg=XGBRegressor(objective='reg:squarederror',tree_method='hist',random_state=SEED,n_jobs=1,verbosity=0,**params)
        reg.fit(xtrain,np.log1p(y))
        p=np.expm1(reg.predict(xvalid))
        return _safe_count(p,model),{'alpha':np.nan,'dispersion_method':'NA','fitted_model':reg,'features':list(xtrain.columns)}
    if pooled:raise ValueError('Poisson/NB2 pooled not in this robustness notebook')
    sx,sv=_design_custom(xtrain,xvalid)
    if model=='Poisson':
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            r=sm.GLM(y,sx,family=sm.families.Poisson()).fit(maxiter=150)
        p=r.predict(sv)
        pearson=float(sum(((y-r.predict(sx))**2)/np.maximum(r.predict(sx),1e-8))/max(len(y)-sx.shape[1],1))
        return _safe_count(p,model),{'alpha':0.,'dispersion_method':'Poisson','poisson_pearson_ratio':pearson}
    if model=='NB2':
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            result=NegativeBinomial(y,sx,loglike_method='nb2').fit(method='bfgs',maxiter=170,disp=0)
        alpha=float(result.params[-1]); converged=bool(result.mle_retvals.get('converged',False))
        if not converged or not np.isfinite(alpha) or alpha<=0:
            raise RuntimeError('NB2 training MLE did not converge; this fold/model was not silently substituted')
        p=result.predict(sv)
        return _safe_count(p,model),{'alpha':alpha,'dispersion_method':'NB2_training_MLE','converged':converged}
    raise ValueError('Unknown model '+model)

def _design_custom(xtrain,xvalid):
    sc=StandardScaler();a=sc.fit_transform(xtrain);b=sc.transform(xvalid)
    return sm.add_constant(a,has_constant='add'),sm.add_constant(b,has_constant='add')

def metrics(data,pred):
    truth=np.asarray(data.Target_Cases_Next_Month,dtype=float)
    err=np.abs(truth-np.asarray(pred,dtype=float))
    return dict(MAE=float(err.mean()),RMSE=float(np.sqrt(np.mean(err**2))), WAPE_pct=float(100*err.sum()/max(truth.sum(),1)),bias=float(np.mean(np.asarray(pred)-truth)))

def score_candidate(source,vyears,model,case_delay=0,weather_delay=0,lag_mode='single',pooled=False):
    predictions=[]; errs=[]; diagnostics=[]
    for valyear in vyears:
        tr=source[source.Year<valyear];va=source[source.Year==valyear]
        if tr.empty or va.empty: continue
        try:
            p,diag=fit_predict(tr,va,model,case_delay,weather_delay,lag_mode,pooled)
            predictions.extend(p.tolist());errs.extend(np.abs(va.Target_Cases_Next_Month.to_numpy()-p).tolist())
            diagnostics.append({k:v for k,v in diag.items() if isinstance(v,(str,float,int,bool,np.floating))})
        except Exception as e:
            ISSUES.append({'stage':'CV','scope':','.join(source.County.unique()),'year':valyear,'candidate':model,'error':str(e)[:250]})
            return float('inf'),[]
    return float(np.mean(errs)) if errs else float('inf'),diagnostics

def _pick_county_candidate(source,inner_years):
    table=[]
    for mod in CANDIDATES:
        val,details=score_candidate(source,inner_years,mod)
        table.append({'candidate':mod,'inner_MAE':val})
    selection=min(table,key=lambda z:z['inner_MAE'])['candidate']
    if not np.isfinite(min(v['inner_MAE'] for v in table)):raise RuntimeError('No viable candidate')
    return selection,table

def run_nested(d):
    rows=[]; inner_records=[]; pooled_records=[]; diag_records=[]
    for outer_year in [2020,2021,2022,2023]:
        inners=list(range(2019,outer_year))
        # Train on years strictly before each inner-validation year.
        for county in COUNTIES:
            ss=d[(d.County==county)&(d.Year<outer_year)]
            chosen,records=_pick_county_candidate(ss,inners)
            for r in records:inner_records.append(dict(outer_year=outer_year,county=county,**r))
            tr=ss; va=d[(d.County==county)&(d.Year==outer_year)]
            p,diag=fit_predict(tr,va,chosen)
            for (_,item),pred in zip(va.iterrows(),p):
                rows.append({'county':county,'target_month':item.Target_Month,'outer_year':outer_year,'strategy':'County_nested_selected','model':chosen,'actual':float(item.Target_Cases_Next_Month),'predicted':float(pred)})
                rows.append({'county':county,'target_month':item.Target_Month,'outer_year':outer_year,'strategy':'Seasonal_baseline','model':'Seasonal','actual':float(item.Target_Cases_Next_Month),'predicted':float(item.Seasonal_Baseline_Cases)})
            diag_records.append(dict(outer_year=outer_year,county=county,chosen=chosen,alpha=diag.get('alpha',np.nan),method=diag.get('dispersion_method','NA')))
        # pooled XGBoost tuned using county-balanced inner-year performance
        candidates=[]
        for mod in XGB_GRID:
            fold_scores=[]
            for vy in inners:
                tr=d[d.Year<vy];va=d[d.Year==vy]
                try:
                    p,_=fit_predict(tr,va,mod,pooled=True)
                    part=va[['County','Target_Cases_Next_Month']].copy();part['p']=p
                    per=part.groupby('County').apply(lambda z:np.sum(abs(z.p-z.Target_Cases_Next_Month))/max(z.Target_Cases_Next_Month.sum(),1), include_groups=False)
                    fold_scores.append(float(per.mean()))
                except Exception as e:
                    ISSUES.append({'stage':'pooled_inner','scope':'all','year':vy,'candidate':mod,'error':str(e)[:250]})
                    fold_scores=[float('inf')];break
            candidates.append({'outer_year':outer_year,'model':mod,'macro_WAPE':float(np.mean(fold_scores))})
        pooled_records.extend(candidates)
        pooled_best=min(candidates,key=lambda z:z['macro_WAPE'])['model']
        pa=d[d.Year==outer_year]
        pp,_=fit_predict(d[d.Year<outer_year],pa,pooled_best,pooled=True)
        for (_,item),pred in zip(pa.iterrows(),pp):
            rows.append({'county':item.County,'target_month':item.Target_Month,'outer_year':outer_year,'strategy':'Pooled_XGB_nested','model':pooled_best,'actual':float(item.Target_Cases_Next_Month),'predicted':float(pred)})
        print('Finished outer',outer_year,'pooled:',pooled_best)
    pr=pd.DataFrame(rows);pr['absolute_error']=(pr.actual-pr.predicted).abs()
    summary=pr.groupby(['county','strategy'],as_index=False).agg(MAE=('absolute_error','mean'),Total_abs_err=('absolute_error','sum'),Total_actual=('actual','sum'))
    summary['WAPE_pct']=100*summary.Total_abs_err/summary.Total_actual
    return pr,summary,pd.DataFrame(inner_records),pd.DataFrame(pooled_records),pd.DataFrame(diag_records)

def freeze_models(d,nested_preds,nested_summary):
    frozen=[]; full_development_search=[]
    for county in COUNTIES:
        cc=nested_summary[nested_summary.county==county]
        best=cc.sort_values(['MAE','strategy']).iloc[0]
        strategy=best.strategy
        train=d[(d.County==county)&(d.Year<=2023)]
        if strategy=='County_nested_selected':
            candidate,records=_pick_county_candidate(train,[2019,2020,2021,2022,2023])
            for r in records:full_development_search.append(dict(scope=county,**r))
        elif strategy=='Pooled_XGB_nested':
            # Pooled XGB chosen via preceding 2020–2023 nested validation only.
            # The final hyperparameter setting is selected in 2019–2023 folds,
            # and no 2024/2025 outcomes are consulted.
            candidates=[]
            for mod in XGB_GRID:
                by_year=[]
                for vy in [2019,2020,2021,2022,2023]:
                    tr=d[d.Year<vy];va=d[d.Year==vy]
                    try:
                        p,_=fit_predict(tr,va,mod,pooled=True)
                        by_year.append(float(np.mean(abs(va[va.County==county].Target_Cases_Next_Month.to_numpy()-p[(va.County==county).to_numpy()]))))
                    except Exception as e:
                        ISSUES.append({'stage':'frozen_pooled_grid','scope':county,'year':vy,'candidate':mod,'error':str(e)[:250]});by_year=[float('inf')];break
                candidates.append({'candidate':mod,'inner_MAE':float(np.mean(by_year))})
            candidate=min(candidates,key=lambda z:z['inner_MAE'])['candidate']
            for r in candidates:full_development_search.append(dict(scope=county,**r))
        else:candidate='Seasonal'
        frozen.append({'county':county,'frozen_strategy':strategy,'frozen_model':candidate,'development_nested_MAE':float(best.MAE)})
    return pd.DataFrame(frozen),pd.DataFrame(full_development_search)

def evaluate_frozen(d,frozen):
    rec=[];diagn=[]
    for year in [2024,2025]:
        for cc in COUNTIES:
            f=frozen[frozen.county==cc].iloc[0]
            tr=d[(d.Year<year)&((d.County==cc) if f.frozen_strategy=='County_nested_selected' else True)]
            va=d[(d.Year==year)&(d.County==cc)]
            pooled=f.frozen_strategy=='Pooled_XGB_nested'
            mod=f.frozen_model
            p,details=fit_predict(tr,va,mod,pooled=pooled)
            for (_,r),x in zip(va.iterrows(),p):
                rec.append({'county':cc,'target_month':r.Target_Month,'year':year,'status':'HELDOUT_2024' if year==2024 else 'PROVISIONAL_2025',
                            'model':mod,'strategy':f.frozen_strategy,'actual':float(r.Target_Cases_Next_Month),'predicted':float(x),
                            'seasonal_prediction':float(r.Seasonal_Baseline_Cases),'early_2024_flag':r.Early_2024_Surveillance_Flag})
            diagn.append({'county':cc,'year':year,'model':mod,'alpha_train_estimate':details.get('alpha'),'poisson_pearson_ratio':details.get('poisson_pearson_ratio'),'method':details.get('dispersion_method','NA')})
    pred=pd.DataFrame(rec)
    pred['absolute_error']=(pred.actual-pred.predicted).abs()
    pred['baseline_absolute_error']=(pred.actual-pred.seasonal_prediction).abs()
    records=[]
    for (county,year),ss in pred.groupby(['county','year']):
        for subset_name, sub in [('All',ss),('Jan_Mar',ss[ss.target_month.str[-2:].astype(int)<=3]),('Apr_Dec',ss[ss.target_month.str[-2:].astype(int)>=4])]:
            records.append({'county':county,'year':year,'period':subset_name,'model':ss.model.iloc[0],'N_months':len(sub),
                            'model_MAE':sub.absolute_error.mean(),'baseline_MAE':sub.baseline_absolute_error.mean(),
                            'model_WAPE_pct':100*sub.absolute_error.sum()/max(sub.actual.sum(),1)})
    return pred,pd.DataFrame(records),pd.DataFrame(diagn)

def scenario_check(d,frozen):
    scenarios=[('Recent_cases_weather',0,0),('Case_1mo_delayed',1,0),('Case_2mo_delayed',2,0),
               ('Cases_1mo_weather_1mo',1,1),('Cases_2mo_weather_1mo',2,1)]
    rows=[]
    for county in COUNTIES:
        f=frozen[frozen.county==county].iloc[0]
        mod=f.frozen_model
        pool=f.frozen_strategy=='Pooled_XGB_nested'
        for name,case_delay,weather_delay in scenarios:
            for year in [2024,2025]:
                tr=d[(d.Year<year) & ((d.County==county) if not pool else True)]
                va=d[(d.Year==year)&(d.County==county)]
                p,_=fit_predict(tr,va,mod,case_delay,weather_delay,pooled=pool)
                for (_,r),a in zip(va.iterrows(),p):
                    rows.append({'county':county,'year':year,'target_month':r.Target_Month,'scenario':name,'case_months_stale':case_delay,
                        'weather_months_stale':weather_delay,'model':mod,'actual':float(r.Target_Cases_Next_Month),'prediction':float(a),
                        'absolute_error':float(abs(r.Target_Cases_Next_Month-a))})
    p=pd.DataFrame(rows)
    m=p.groupby(['county','year','scenario','case_months_stale','weather_months_stale'],as_index=False).agg(MAE=('absolute_error','mean'),N=('actual','count'),total_error=('absolute_error','sum'),total_cases=('actual','sum'))
    m['WAPE_pct']=100*m.total_error/m.total_cases
    return p,m

def lag_ablation(d):
    # Predeclare 3 groups; choose ONLY from development folds 2019-2023.
    choices=[];details=[];holdouts=[]
    for county in COUNTIES:
        s=d[d.County==county]
        options=[('current',0,'single'),('prior_month',1,'single'),('two_months_prior',2,'single'),('current_plus_prior',0,'plus_lag1')]
        cand=[]
        for opt,wd,lg in options:
            all_pred=[];all_true=[]
            for vyear in [2019,2020,2021,2022,2023]:
                tr=s[s.Year<vyear];va=s[s.Year==vyear]
                pp,_=fit_predict(tr,va,'XGB_small',weather_delay=wd,lag_mode=lg)
                all_pred.extend(pp);all_true.extend(va.Target_Cases_Next_Month.to_numpy(dtype=float))
            mae=float(np.mean(abs(np.asarray(all_pred)-np.asarray(all_true))))
            cand.append(dict(county=county,lag_group=opt,weather_delay=wd,feature_group=lg,development_MAE=mae))
        details.extend(cand)
        winning=min(cand,key=lambda z:z['development_MAE'])
        choices.append(winning)
        train=s[s.Year<=2023];test=s[s.Year==2024]
        pr,_=fit_predict(train,test,'XGB_small',weather_delay=winning['weather_delay'],lag_mode=winning['feature_group'])
        for (_,r),v in zip(test.iterrows(),pr):holdouts.append({'county':county,'target_month':r.Target_Month,'chosen_dev_lags':winning['lag_group'],'actual':r.Target_Cases_Next_Month,'predicted':v,'absolute_error':abs(r.Target_Cases_Next_Month-v)})
    return pd.DataFrame(details),pd.DataFrame(choices),pd.DataFrame(holdouts)

def county_transfer(d):
    # Explicit geographically out-of-county exploratory test, no hidden-county targets at train time.
    # The fit uses only 2016-2019 on OTHER counties; 2020-2023 held county is unseen in training.
    results=[]
    for hold in COUNTIES:
        train=d[(d.County!=hold)&(d.Year<=2019)];valid=d[(d.County==hold)&(d.Year.between(2020,2023))]
        xtr=features(train).drop(columns=['case_log','seasonal_log'])
        xva=features(valid).drop(columns=['case_log','seasonal_log'])
        # case-history normalization enables transfer without using an unseen county dummy.
        xtr['case_vs_season']=np.log1p(train.VF_cases_current.values)-np.log1p(train.Seasonal_Baseline_Cases.values)
        xva['case_vs_season']=np.log1p(valid.VF_cases_current.values)-np.log1p(valid.Seasonal_Baseline_Cases.values)
        ry=np.log1p(train.Target_Cases_Next_Month.values)-np.log1p(train.Seasonal_Baseline_Cases.values)
        m=XGBRegressor(objective='reg:squarederror',random_state=SEED,tree_method='hist',n_jobs=1,verbosity=0,**XGB_GRID['XGB_small'])
        m.fit(xtr,ry)
        pp=np.maximum(0,np.expm1(m.predict(xva)+np.log1p(valid.Seasonal_Baseline_Cases.values)))
        for (_,r),v in zip(valid.iterrows(),pp):
            results.append({'county_held_out':hold,'target_month':r.Target_Month,'actual':r.Target_Cases_Next_Month,'transfer_prediction':v,'baseline_prediction':r.Seasonal_Baseline_Cases,'absolute_error':abs(r.Target_Cases_Next_Month-v)})
    pr=pd.DataFrame(results)
    smry=pr.groupby('county_held_out',as_index=False).apply(lambda z:pd.Series({'Transfer_MAE':z.absolute_error.mean(),'Baseline_MAE':np.abs(z.actual-z.baseline_prediction).mean(),'Months':len(z)}),include_groups=False).reset_index(drop=True)
    return pr,smry



In [ ]:
import zipfile, shutil
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

out=Path(OUTPUT_DIR)
if out.exists():
    shutil.rmtree(out)
out.mkdir(exist_ok=True)
d=read_input()

# Source audit: does not verify external dates, station geographies, or provisional revisions.
audit=[]
for county,s in d.groupby('County'):
    expected_fips={'Maricopa':'04013','Pinal':'04021','Pima':'04019'}[county]
    actual_fips=set(s.County_FIPS.astype(str).str.zfill(5))
    if actual_fips!={expected_fips}:raise ValueError(f'{county}: county FIPS mismatch: {actual_fips}')
    if s.Target_Month.nunique()!=119:raise ValueError(f'{county}: missing month')
    audit.append({'county':county,'FIPS':expected_fips,'forecast_rows':len(s),'first_target':s.Target_Month.min(),'last_target':s.Target_Month.max(),
         'year2024_cases':int(s.loc[s.Year==2024,'Target_Cases_Next_Month'].sum()),'year2025_cases_provisional':int(s.loc[s.Year==2025,'Target_Cases_Next_Month'].sum()),
         'geography':str(s.Source_geography_note.iloc[0])})
audit_df=pd.DataFrame(audit)
assert dict(zip(audit_df.county,audit_df.year2024_cases))=={'Maricopa':10920,'Pima':1614,'Pinal':1484}
print('Input audit: 357 forecast rows; 119 per county; cases match 2024 ADHS workbook totals.')

# 1. Truly chronological inner/outer development validation.
nested,dev_summary,inner,pooled,fit_detail=run_nested(d)
frozen,freeze_search=freeze_models(d,nested,dev_summary)
print('\nDevelopment-chosen 2024 models (no 2024-target tuning):')
print(frozen.to_string(index=False))

# 2. Frozen 2024 and 2025 experiments.
held,held_metrics,fitted_diagnostics=evaluate_frozen(d,frozen)

# 3. Dispersion MLE and Poisson vs NB2 evaluated as predeclared diagnostics, not test-driven selection.
disp=[]
for county in COUNTIES:
    tr=d[(d.County==county)&(d.Year<=2023)]
    va=d[(d.County==county)&(d.Year==2024)]
    for model in ['Poisson','NB2']:
        pp,info=fit_predict(tr,va,model)
        met=metrics(va,pp)
        disp.append({'county':county,'model':model,'train_end':'2023-12','2024_MAE':met['MAE'],
           '2024_RMSE':met['RMSE'],'2024_WAPE_pct':met['WAPE_pct'],
           'dispersion_alpha_train_MLE':info.get('alpha'),'alpha_fit_method':info.get('dispersion_method'),
           'Poisson_train_Pearson_overdispersion_ratio':info.get('poisson_pearson_ratio')})
disp=pd.DataFrame(disp)

# 4. No-change 2024 sensitivity analysis already in held_metrics: All / Jan-Mar / Apr-Dec.

# 5. Controlled, hypothetical availability stress tests.
delay_preds,delay_metrics=scenario_check(d,frozen)

# 6. Development-only environmental lag ablation; no held-out tuning.
lag_dev,lag_winner,lag_2024=lag_ablation(d)

# 7. Out-of-county temporal transfer, with explicitly stated held-county case-history requirement.
transfer,transfer_summary=county_transfer(d)

# 8. Case-report anomaly robustness for provisional 2025. Fixed model selection, two training sets.
q1_rows=[]
for county in COUNTIES:
    ff=frozen[frozen.county==county].iloc[0]
    pooled_bool=ff.frozen_strategy=='Pooled_XGB_nested'
    train_mask=(d.Year<2025) & (True if pooled_bool else (d.County==county))
    test=d[(d.Year==2025)&(d.County==county)]
    for variant in ['include_2024_Jan_Mar','omit_2024_Jan_Mar_targets']:
        tr=d[train_mask].copy()
        if variant.startswith('omit'):
            tr=tr[~((tr.Year==2024)&(tr.Month<=3))]
        p,_=fit_predict(tr,test,ff.frozen_model,pooled=pooled_bool)
        for (_,r),z in zip(test.iterrows(),p):
            q1_rows.append({'county':county,'target_month':r.Target_Month,'training_variant':variant,'prediction':z,'actual':r.Target_Cases_Next_Month,
               'absolute_error':abs(r.Target_Cases_Next_Month-z)})
q1_pred=pd.DataFrame(q1_rows)
q1_metric=q1_pred.groupby(['county','training_variant'],as_index=False).agg(MAE=('absolute_error','mean'))

# 9. SHAP on 2024 predictions from XGBoost; for Maricopa this explains a challenger, not chosen NB2.
shap_rank=[];shap_meta=[];shap_contrib=[];shap_add=[]
try:
 import shap
 shap_version=shap.__version__
 for county in COUNTIES:
    f=frozen[frozen.county==county].iloc[0]
    if f.frozen_model.startswith('XGB'):
        chosen=f.frozen_model
        is_pooled=f.frozen_strategy=='Pooled_XGB_nested'
        kind='Frozen selected XGBoost'
    else:
        chosen='XGB_small';is_pooled=False
        kind='Exploratory XGBoost challenger, NOT frozen selected model'
    train=d[(d.Year<=2023)&(True if is_pooled else (d.County==county))]
    test=d[(d.Year==2024)&(d.County==county)]
    xtrain=features(train,pooled=is_pooled)
    xtest=features(test,pooled=is_pooled)
    model=XGBRegressor(objective='reg:squarederror',tree_method='hist',random_state=SEED,n_jobs=1,verbosity=0,**XGB_GRID[chosen])
    model.fit(xtrain,np.log1p(train.Target_Cases_Next_Month.to_numpy()))
    expl=shap.TreeExplainer(model)
    sv=expl(xtest)
    values=np.asarray(sv.values)
    base=np.asarray(sv.base_values).reshape(-1)
    check=np.asarray(model.predict(xtest))
    resid=float(np.max(abs(values.sum(axis=1)+base-check)))
    shap_add.append({'county':county,'model':chosen,'max_abs_additivity_discrepancy_log1p':resid})
    shap_meta.append({'county':county,'model':chosen,'role':kind,'basis':'2024 feature values; model fit on <=2023 target months','SHAP_output_scale':'log1p(monthly cases)','SHAP_package':shap_version})
    ranks=pd.DataFrame({'feature':xtest.columns,'mean_abs_SHAP_log1p':np.mean(abs(values),axis=0)}).sort_values('mean_abs_SHAP_log1p',ascending=False)
    for r in ranks.itertuples():shap_rank.append({'county':county,'feature':r.feature,'mean_abs_SHAP_log1p':r.mean_abs_SHAP_log1p,'model_role':kind})
    for row_idx,(_,rr) in enumerate(test.iterrows()):
        for feature,score in zip(xtest.columns,values[row_idx]):
            shap_contrib.append({'county':county,'target_month':rr.Target_Month,'feature':feature,'SHAP_value_log1p':float(score), 'model_role':kind})
    fig,ax=plt.subplots(figsize=(8,5))
    z=ranks.head(10).iloc[::-1]
    ax.barh(z['feature'],z['mean_abs_SHAP_log1p'])
    ax.set_title(f'{county} 2024 SHAP | {kind}')
    ax.set_xlabel('Mean absolute SHAP contribution (log1p cases)')
    fig.tight_layout();fig.savefig(out/f'SHAP_{county}.png',dpi=160);plt.close(fig)
except Exception as e:
 ISSUES.append({'stage':'SHAP','scope':'all','year':2024,'candidate':'XGBoost','error':str(e)[:250]})
 print('SHAP failed; main result tables still valid:',str(e))
shap_rank_df=pd.DataFrame(shap_rank,columns=['county','feature','mean_abs_SHAP_log1p','model_role'])
shap_meta_df=pd.DataFrame(shap_meta,columns=['county','model','role','basis','SHAP_output_scale','SHAP_package'])
shap_val_df=pd.DataFrame(shap_add,columns=['county','model','max_abs_additivity_discrepancy_log1p'])
shap_contrib_df=pd.DataFrame(shap_contrib,columns=['county','target_month','feature','SHAP_value_log1p','model_role'])

# Equal-county metric (mean of county MAEs, not total error / all cases).
macro=held_metrics[(held_metrics.period=='All')].groupby('year',as_index=False).agg(Macro_avg_county_MAE=('model_MAE','mean'),Macro_avg_baseline_MAE=('baseline_MAE','mean'),Macro_avg_county_WAPE_pct=('model_WAPE_pct','mean'))
macro['Important']='Equal average over 3 counties; not pooled statewide MAE'

# figures: count errors and sensitivity
h2024=held_metrics[(held_metrics.year==2024)&(held_metrics.period=='All')].set_index('county').loc[COUNTIES]
fig,ax=plt.subplots(figsize=(9,5));xs=np.arange(3);w=.37
ax.bar(xs-w/2,h2024.baseline_MAE,width=w,label='Seasonal baseline')
ax.bar(xs+w/2,h2024.model_MAE,width=w,label='Frozen selected model')
ax.set_xticks(xs,COUNTIES);ax.set_ylabel('2024 MAE: cases per month (lower is better)')
ax.set_title('Holdout performance; settings chosen using development data only')
ax.legend();fig.tight_layout();fig.savefig(out/'2024_Comparison.png',dpi=170);plt.close(fig)
fig,ax=plt.subplots(figsize=(9,5));dy=dev_summary.pivot(index='county',columns='strategy',values='MAE').reindex(COUNTIES)
for j,col in enumerate(dy.columns):ax.plot(xs,dy[col],marker='o',label=col)
ax.set_xticks(xs,COUNTIES);ax.set_ylabel('2020–2023 nested development MAE');ax.set_title('Nested walk-forward development evaluation')
ax.legend(fontsize=8);fig.tight_layout();fig.savefig(out/'Development_Nested_Comparison.png',dpi=170);plt.close(fig)

exports={
 '00_Input_and_Geography_Audit':audit_df,
 '01_Development_Nested_Predictions':nested,
 '02_Development_Strategy_Comparison':dev_summary,
 '03_Inner_Year_Search':inner,
 '04_Pooled_XGBoost_Inner_Search':pooled,
 '05_Frozen_2024_Selections':frozen,
 '06_Final_Development_Search':freeze_search,
 '07_2024_2025_Frozen_Predictions':held,
 '08_2024_2025_Heldout_Metrics':held_metrics,
 '09_Poisson_vs_NB2_Dispersion':disp,
 '10_Model_Training_Diagnostics':fitted_diagnostics,
 '11_Reporting_Delay_Predictions':delay_preds,
 '12_Reporting_Delay_Metrics':delay_metrics,
 '13_Environmental_Lag_Dev':lag_dev,
 '14_Environmental_Lag_Selections':lag_winner,
 '15_Environmental_Lag_2024_Test':lag_2024,
 '16_Cross_County_Transfer':transfer_summary,
 '17_Cross_County_Transfer_Monthly':transfer,
 '18_2025_Q1_Training_Sensitivity':q1_metric,
 '19_2025_Q1_Sensitivity_Monthly':q1_pred,
 '20_SHAP_Rankings':shap_rank_df,
 '21_SHAP_Model_Metadata':shap_meta_df,
 '22_SHAP_Additivity':shap_val_df,
 '23_SHAP_Monthly_Contributions':shap_contrib_df,
 '24_County_Balanced_Macro':macro,
 '25_Fit_Issues':pd.DataFrame(ISSUES,columns=['stage','scope','year','candidate','error'])
}
for name,table in exports.items():table.to_csv(out/(name+'.csv'),index=False)

# Reproducible research instructions and nonclaims.
readme=f'''VALLEY FEVER | TIER 2 FINAL ROBUSTNESS CHECKS
Input: VF_ThreeCounty_Stage2_Model_Input.csv | Rows: {len(d)} | Counties: Maricopa, Pinal, Pima

A. NESTED CHRONOLOGICAL MODEL SELECTION
Outer years: 2020, 2021, 2022, 2023. For outer year Y, inner validation years are 2019..Y-1.
For every inner year K: training strictly precedes K. Outer year target outcomes are unseen during its inner selection.
County strategy compares seasonal baseline, Poisson, NB2 with training-MLE dispersion, and three fixed XGBoost settings.
Pooled strategy compares three XGBoost settings by equal-county WAPE in inner folds.
After development, freeze county-vs-pooled strategy and hyperparameters before scoring 2024. 2025 is provisional.
2024 January-March ADHS laboratory-test concern: cases are NOT modified; evaluate Apr-Dec separately.

B. NEGATIVE BINOMIAL AND POISSON
NB2 alpha is estimated by maximum likelihood separately using training data; nonconvergent fits are reported.
The Poisson training Pearson chi-square / residual-df is a rough overdispersion diagnostic, not proof of NB validity.
The 2024 Poisson-vs-NB table is a predeclared diagnostic, NOT used to choose models for that test.

C. SHAP + LAG ABLATION
SHAP on XGBoost predicts log(1+cases). SHAP values are in log-count units and NOT causal effects.
Maricopa's SHAP XGBoost is a CHALLENGER if its frozen winning model is Negative Binomial.
Lag choices use development target years 2019-2023 only; evaluated on the 2024 set afterward.

D. COUNTY BALANCE + SPATIAL LIMITATIONS
Scores are reported by county and as an equal-county average. Do not compare one unit of Maricopa MAE with one unit of Pima MAE without context.
The leave-one-county-out test STILL uses the withheld county's historic case features and seasonal baseline. It tests transfer of environmental correction patterns, NOT zero-shot epidemiology or ZIP forecasts.
PRISM values are from county centroids; AZMet is a small station-network proxy; EPA monitor density differs by county.
County-level labels do NOT validate ZIP-code or wind-edge graph forecasting.

E. CASE/ENVIRONMENT AVAILABILITY STRESS TEST
'Stale by 1 or 2 months' shifts which case/weather inputs are included in BOTH training and prediction.
These are simulated reporting/publication delays, NOT historical release snapshots.
Actual availability remains UNVERIFIED: monthly ADHS event-date report publication and revision logs; EPA and PRISM release calendars; AZMet publication gaps.
Therefore the reported model is a retrospective research prototype and NOT a validated operational early-warning system.

F. 2025 PROVISIONAL SENSITIVITY
Refit final frozen models with or without Jan-Mar 2024 rows in training; the case-history input features in later rows may still incorporate those months. This is not a full decontamination.

G. INTERPRETATION
2024 MAEs from this strict nested experiment need not match earlier Stage 2 experiments: the validation protocol, feature selection, NB alpha estimation, and tuning search differ.
Do not retrospectively choose a new winner using 2024. Treat 2024 as a final audit in this run.

Frozen 2024 selections:
{frozen.to_string(index=False)}

2024 held-out model vs baseline, MAE:
{held_metrics[(held_metrics.year==2024)&(held_metrics.period=='All')][['county','model','model_MAE','baseline_MAE']].to_string(index=False)}

OUTPUTS
CSV 00-25: input audit, nested search, frozen performance, Poisson vs NB, lags, delays, county transfer, SHAP, 2025 robustness, and fit errors.
Images: 2024_Comparison, Development_Nested_Comparison, SHAP_*.png.
A separately supplied summary workbook is created from these CSV outputs for convenience.

Research use only. No measured public-health release timing and no patient-level or ZIP-level data.
'''
(out/'README_RESULTS.txt').write_text(readme,encoding='utf-8')
zip_path=Path('VF_Tier2_Final_Results.zip')
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as zp:
    for file in sorted(out.iterdir()):
        if file.is_file():zp.write(file,arcname=file.name)
assert zipfile.ZipFile(zip_path).testzip() is None
print('\nSUCCESS',zip_path,round(zip_path.stat().st_size/1024),'KB')
print(held_metrics[(held_metrics.year==2024)&(held_metrics.period=='All')][['county','model','model_MAE','baseline_MAE']].to_string(index=False))
print('Train fit / SHAP warnings:',len(ISSUES))


## Part C — Build one results Excel and one downloadable ZIP

The workbook below puts the most useful Checkpoint 2 results in named tabs. All detailed Tier 2 CSVs and plots remain in the original final Tier 2 output ZIP. The ZIP downloaded in the last cell includes **both** Tier 1 and Tier 2 results.

In [ ]:
# STEP 5 — Produce the final Checkpoint 2 Excel summary and a combined results ZIP.
import pandas as pd, numpy as np, zipfile, shutil
from pathlib import Path

excel_name='VF_Checkpoint2_Tier1_Tier2_Results.xlsx'
h24=held_metrics[(held_metrics.year==2024)&(held_metrics.period=='All')].copy()
main=h24.merge(frozen[['county','frozen_strategy','frozen_model','development_nested_MAE']],on='county')
main['MAE_reduction_pct']=100*(main['baseline_MAE']-main['model_MAE'])/main['baseline_MAE']
main=main[['county','frozen_model','frozen_strategy','N_months','baseline_MAE','model_MAE','MAE_reduction_pct','development_nested_MAE']]
main=main.sort_values('county').reset_index(drop=True)

sensitivity=held_metrics[(held_metrics.year==2024)&(held_metrics.period.isin(['All','Apr_Dec']))].copy()
sensitivity['MAE_reduction_pct']=100*(sensitivity.baseline_MAE-sensitivity.model_MAE)/sensitivity.baseline_MAE
sensitivity=sensitivity.sort_values(['county','period'])

shap_top=(shap_rank_df.sort_values(['county','mean_abs_SHAP_log1p'],ascending=[True,False])
          .groupby('county',as_index=False).head(5))
provisional=held_metrics[(held_metrics.year==2025)&(held_metrics.period=='All')].copy()
# Original saved-run values are shown ONLY for provenance, never used in training or model selection.
original_saved=pd.DataFrame([
 {'county':'Maricopa','saved_model':'NB2','saved_2024_MAE':132.13810497258962,'saved_baseline_MAE':240.16666666666666},
 {'county':'Pinal','saved_model':'XGB_medium','saved_2024_MAE':26.524394353230793,'saved_baseline_MAE':40.833333333333336},
 {'county':'Pima','saved_model':'XGB_regularized','saved_2024_MAE':28.321238835652668,'saved_baseline_MAE':32.333333333333336},
])
from importlib.metadata import version
software=pd.DataFrame([{'Package':p,'Version_in_this_run':version(p)} for p in
                       ['numpy','pandas','scikit-learn','xgboost','statsmodels','shap']])
notes=[
 ('Saved versus rerun','Archived October 15 results may differ from this rerun. Original package did not record pinned versions; do not mix values.'),
 ('Purpose','Reproducible Checkpoint 2 demonstration, not a clinical early-warning product'),
 ('Data','ADHS event-date monthly cases, EPA PM10, PRISM county-centroid climate, AZMet station weather'),
 ('Coverage','Maricopa, Pinal, Pima; Jan 2015-Dec 2025 source, 396 county-month and 357 forecast rows'),
 ('Tier 1','Seasonal naive, Ridge, nonlinear boosting; chronological 2019-2023 development'),
 ('Tier 2','Nested development 2020-2023, frozen selected NB2 / pooled XGBoost; 2024 audit'),
 ('2024 Q1','Reported laboratory test-kit concern; do not edit ADHS case counts; Apr-Dec scores exclude only flagged target months'),
 ('2025','Provisional cases; show separately, not as an untouched benchmark'),
 ('Data availability','Case reporting and environmental release-date history not reconstructed; delays are simulated'),
 ('SHAP','Explains XGBoost log(1+cases) predictions, not causality; Maricopa SHAP is a challenger, not selected NB2'),
 ('Model selection','2024 examined in earlier project experiments; not a never-before-seen independent test'),
 ('Tier 3','Spatial GNN/STGCN not trained; no ZIP-level case targets'),
 ('SARIMA','Mentioned in proposal but not trained in this package; confirm with professor if mandatory'),
]
workbook_sheets={
 '2024_MAIN':main,
 '2024_SENSITIVITY':sensitivity,
 'TIER1_METRICS':all_metrics,
 'TIER2_METRICS':held_metrics,
 'MODEL_SELECTION':frozen,
 '2024_MONTHLY':held[held.year==2024],
 'DEVELOPMENT':dev_summary,
 '2025_PROVISIONAL':provisional,
 'SHAP_TOP':shap_top,
 'LAG_SELECTION':lag_winner,
 'LAG_DEVELOPMENT':lag_dev,
 'DELAY_METRICS':delay_metrics,
 'DATA_AUDIT':audit_df,
 'FIT_ISSUES':pd.DataFrame(ISSUES,columns=['stage','scope','year','candidate','error']),
 'READ_ME':pd.DataFrame(notes,columns=['Topic','Explanation']),
 'ORIGINAL_SAVED_2024':original_saved,
 'SOFTWARE_VERSIONS':software,
}
with pd.ExcelWriter(excel_name,engine='xlsxwriter') as writer:
    book=writer.book
    header_fmt=book.add_format({'bold':True,'font_color':'#FFFFFF','bg_color':'#163A53','border':0,'text_wrap':True,'valign':'vcenter'})
    decimal_fmt=book.add_format({'num_format':'0.00'})
    for sheet_name,table in workbook_sheets.items():
        table.to_excel(writer,sheet_name=sheet_name,index=False)
        ws=writer.sheets[sheet_name]
        ws.freeze_panes(1,0)
        ws.set_row(0,31,header_fmt)
        ws.autofilter(0,0,max(len(table),1),max(len(table.columns)-1,0))
        for col_idx,col_name in enumerate(table.columns):
            vals=table[col_name].dropna().astype(str).head(120)
            width=min(45,max(14,len(str(col_name))+2,max((len(x) for x in vals),default=8)+1))
            ws.set_column(col_idx,col_idx,width)
            if pd.api.types.is_float_dtype(table[col_name]):ws.set_column(col_idx,col_idx,width,decimal_fmt)
        if sheet_name=='READ_ME':ws.set_column(1,1,105)
print('Excel ready:',excel_name)

readme="""CHECKPOINT 2 — THREE-COUNTY VALLEY FEVER REPRODUCIBLE RESULTS
Run VF_Checkpoint2_Tier1_Tier2_One_Colab.ipynb with VF_Checkpoint2_Input_Three_Counties.xlsx.
Part A reproduces the original combined Tier 1 models.
Part B runs the original final Tier 2 notebook (development-only nested selection, 2024/2025, SHAP, lags, delays).
Part C writes one Excel summary and this ZIP.
Show 2024_MAIN, 2024_SENSITIVITY, MODEL_SELECTION, and SHAP_TOP in Checkpoint 2.
Do not mix the earlier Maricopa-only Ridge demo with the combined Tier 1 results.
2024 is retrospective and previously inspected; 2025 is provisional. Tier 3 was not trained.
No original ADHS cases were edited. Reporting delays are simulated, not actual publication logs.
"""
Path('VF_Checkpoint2_READ_ME.txt').write_text(readme,encoding='utf-8')
zip_name='VF_Checkpoint2_All_Results.zip'
with zipfile.ZipFile(zip_name,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    z.write(excel_name)
    z.write('VF_Checkpoint2_READ_ME.txt')
    for p in sorted(TIER1_DIR.glob('*.csv')):z.write(p,arcname='Tier1_CSVs/'+p.name)
    z.write('VF_Tier2_Final_Results.zip',arcname='Tier2_Final_Results.zip')
assert zipfile.ZipFile(zip_name).testzip() is None
print('Combined results ready:',zip_name,round(Path(zip_name).stat().st_size/1024),'KB')
print('Software versions used in this run:')
print(software.to_string(index=False))
print('2024 selected models and MAE:')
print(main.to_string(index=False))
print('Archived 2024 run (reference only, not used for selection):')
print(original_saved.to_string(index=False))
try:
    from google.colab import files
    files.download(zip_name)
except ImportError:
    print('Download manually from working folder:',zip_name)